In [ ]:
import sys
sys.path.append('../../utils')
from functions import * 

In [ ]:
from importlib import reload
import sys
import scipy as sp
from scipy.special import expit, logit

import matplotlib.pyplot as plt 

# Path to the Leaflet repository
PATH_TO_LEAFLET_REPO = '/gpfs/commons/home/kisaev/Leaflet/src/beta-binomial-mix/'
sys.path.append(PATH_TO_LEAFLET_REPO)

In [ ]:
import cell_state_asign_consistency
reload  (cell_state_asign_consistency)

import betabinomo_mix_singlecells
reload (betabinomo_mix_singlecells)

import  load_cluster_data
reload (load_cluster_data)

from betabinomo_mix_singlecells import *
#reload(betabinomo_mix_singlecells)
from cell_state_asign_consistency import *
#reload(cell_state_asign_consistency)
import torch
import sklearn.manifold 
import plotnine as p9
import time
# indicate plot should be small 4 by 4
import plotnine as p9
from plotnine import ggplot, geom_point, aes, stat_smooth, facet_wrap, geom_violin, theme, element_blank, geom_text, geom_bar, geom_hline
import plotnine
from tqdm import tqdm
plotnine.options.figure_size = (4, 4)
import seaborn as sns
sns.set_theme(style="whitegrid")

### Some utility functions 


In [ ]:
# write function that takes in Cluster name 
def check_SS_cluster(cluster_name):
    
    juncs_c = junc_info[junc_info["Cluster"] == cluster_name]
    
    # keep only rows where either start or end appear twice
    s = np.array(juncs_c[juncs_c.duplicated(subset=['start'])].start.unique())
    e = np.array(juncs_c[juncs_c.duplicated(subset=['end'])].end.unique())
    juncs_c = juncs_c[(juncs_c["start"].isin(s)) | (juncs_c["end"].isin(e))]

    # if num rows in juncs_c is 3 then return cluster name 
    if len(juncs_c == 3):
        # confirm cluster has non zero counts in both cell types 
        num_celltypes = len(summarized_data[summarized_data.Cluster == cluster_name].cell_type.unique())
        if num_celltypes > 1:
            # confirm that each junction has non zero counts in both cell types
            counts_juncs = summarized_data[summarized_data.Cluster == cluster_name].junction_id.value_counts()
            counts_juncs = counts_juncs[counts_juncs > 1]
            num_cells=len(counts_juncs.index.unique())
            if num_cells == 3:
                return cluster_name
    else:
        pass

In [ ]:
def simulate_junc_counts(cluster_counts, junc_info, cell_types=None, psi_prior_shape1=0.5, psi_prior_shape2=0.5):
    
    """Simulate junc counts while keeping the cluster counts of observed data. 
    
    Args: 
        cluster_counts: scipy coo_matrix. 
        cell_types: pandas Categorical series of pre-defined cell types to use for simulations 
        psi_prior_shape1: float.
        psi_prior_shape2: float.
    Returns:
        sim_junc_counts: scipy coo_matrix. 
        cell_type_labels: numpy array of cell type labels. 
        cell_type_psi: numpy array of cell type specific PSI values.
    """
    
    N, P = cluster_counts.shape  # number of cells, number of junctions
     
    # use real cell types labels to represent intron clusters being higher/lower in specific cell types 
    print("Using pre-defined cell types!")
    cell_type_labels = cell_types.cat.codes.to_numpy()
    K = len(cell_types.cat.categories)  # number of cell types
    
    print(N, P, K, len(cell_types))
    
    # number of intron clusters 
    num_clusters = len(junction_ids_conversion.Cluster.unique())

    # label clusters as positive or negative by sampling 
    cluster_labels = np.random.choice([0, 1], size=num_clusters)

    # make a mapping of Cluster ID to cluster_labels
    cluster_labels_dict = dict(zip(junction_ids_conversion.Cluster.unique(), cluster_labels))

    # initiate empty dataframe cell_type_psi_df to which we will append the simulated PSI values for each junction in each cell type
    cell_type_psi_df = pd.DataFrame()

    for clust in junction_ids_conversion.Cluster.unique():
        print(clust)
        # get cluster label
        clust_label = cluster_labels_dict[clust]
        # get junctions in cluster and order them by start and end 
        juncs_c = junc_info[junc_info["Cluster"] == clust]
        # order juncs_c by start and end
        juncs_c = juncs_c.sort_values(by=['start', 'end'])
        # assign J1, J2, and J3 to junctions where J1+J2 correspond to exon inclusion and J3 corresponds to exon skipping
        juncs_c["junction"] = ["J1", "J3", "J2"] 
        num_juncs = len(juncs_c)
        
        if clust_label == 0: 
            # sample PSI values for each junction in each cell type via pre-defined beta distributions
            probs = torch.distributions.beta.Beta(psi_prior_shape1, psi_prior_shape2).sample([num_juncs, K]) 
            # get J3 prob 
            probs[1,] = probs[1,1]
            probs[0,] = 1-probs[1,1]
            probs[2,] = 1-probs[1,1]
            # convert probs to dataframe 
            probs_df = pd.DataFrame(probs.numpy())
            # add junction_id_index column to probs_df
            probs_df["junction_id_index"] = juncs_c["junction_id_index"].values
            probs_df["sample_label"] = "negative"
            # appent probs_df to cell_type_psi_df
            cell_type_psi_df = cell_type_psi_df.append(probs_df)

        elif clust_label == 1:
            probs = torch.distributions.beta.Beta(psi_prior_shape1, psi_prior_shape2).sample([num_juncs, K]) 
            # get J3 prob
            J3_prob = probs[1,]
            probs[0,] = 1-J3_prob
            probs[2,] = 1-J3_prob
            probs_df = pd.DataFrame(probs.numpy())
            probs_df["junction_id_index"] = juncs_c["junction_id_index"].values
            probs_df["sample_label"] = "positive"
            cell_type_psi_df = cell_type_psi_df.append(probs_df)

    cell_type_psi_df = cell_type_psi_df.sort_values(by=['junction_id_index'])
    cell_type_psi = torch.tensor(cell_type_psi_df[[0,1]].to_numpy()) #should specify K columns insted of "0,1"
    # simulate PSI for each junction in each cell type via pre-defined beta distributions
    #cell_type_psi = torch.distributions.beta.Beta(psi_prior_shape1, psi_prior_shape2).sample([P, K]) 
    print("Done simulating PSI!")

    # use real cluster counts to simulate junc counts with binomial distribution
    sim_junc_counts = cluster_counts.copy() 

    sim_junc_counts.data = torch.distributions.binomial.Binomial( 
         total_count=torch.tensor(cluster_counts.data), 
         probs=cell_type_psi[
             cluster_counts.col, 
             cell_type_labels[cluster_counts.row] #use cell type of each cell to get the corresponding psi
         ]
    ).sample().numpy()
    
    print("Done simulating junc counts!")
    
    return sim_junc_counts, cell_type_labels, cell_type_psi, cell_type_psi_df

### Settings and Load data

In [ ]:
torch.manual_seed(42)

# set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

K = 5 # set to very high number 

float_type = { 
    "device" : device, 
    "dtype" : torch.float, # save memory
}

hypers = {
    "eta" : 1./K, 
    "alpha_prior" : 1, # karin had 0.65 
    "pi_prior" : 1
}

In [ ]:
hypers["eta"]

### Load data

In [ ]:
# this folder contains input data for each tissue cell type sample
input_files_folder = '/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaMurisBrain/BBmixture_simulation_annot/'

# read each file in the input_files_folder (h5) files and concatenate them all into the summarized_data object 
files = os.listdir(input_files_folder)
df_list = []
for file in files:
    if file.endswith(".h5"):
        path_with_quotes = input_files_folder + file
        fixed_path = path_with_quotes.replace("'", "")
        df = pd.read_hdf(fixed_path, 'df')
        df_list.append(df)
    else:
        pass
# concatenate all dataframes input file
summarized_data = pd.concat(df_list, ignore_index=True)

In [ ]:
print(len(summarized_data.cell_id.unique())) # num cells 
print(len(summarized_data.junction_id.unique())) # num junctions
print(len(summarized_data.Cluster.unique())) # num Cluster

In [ ]:
# check if any singletons clusters exist in the data 
clusts_unique = summarized_data[["junction_id", "Cluster"]].drop_duplicates()
clusts_unique = clusts_unique.Cluster.value_counts()
clusts_unique.sort_values(ascending=True)

In [ ]:
summarized_data.head()

In [ ]:
final_data, coo_counts_sparse, coo_cluster_sparse, cell_ids_conversion, junction_ids_conversion = load_cluster_data.load_cluster_data(
    input_folder = input_files_folder, celltypes=['Brain_Non-Myeloid_oligodendrocyte', 'Brain_Myeloid_microglial_cell'], max_intron_count=1500) 

In [ ]:
# add gene_id to junction_ids_conversion via summarized_data 
juncs_genes = summarized_data[['junction_id', 'gene_id']].drop_duplicates()
# merge junctions with genes
junction_ids_conversion = junction_ids_conversion.merge(juncs_genes, on="junction_id")
junction_ids_conversion.head()

In [ ]:
# how many unique genes start off with? 
print(len(junction_ids_conversion.gene_id.unique()))

In [ ]:
cells_unique = summarized_data[["cell_id", "cell_type"]].drop_duplicates()
cells_unique.cell_type.value_counts()

In [ ]:
# still do preprocessing in scipy
import scipy.sparse as sp
indices = (final_data.cell_id_index, final_data.junction_id_index)
indices_np = np.stack(indices)
junc_counts = sp.coo_matrix((final_data.junc_count, indices))
cluster_counts = sp.coo_matrix((final_data.cluster_count, indices))

### For simulating data, let's only keep clusters that have exon skipping event, so cluster with three junctions 

In [ ]:
# SS are shared between end of J1 and start of J2 and end of J2 and start of J3
junc_info = junction_ids_conversion[["junction_id", "Cluster", "junction_id_index"]].drop_duplicates()

# get number of junctions in each cluster first 
cluster_junc_counts = junc_info.groupby(["Cluster"]).agg({"junction_id": "count"}).reset_index()
clusts_keep = cluster_junc_counts[cluster_junc_counts["junction_id"] == 3 ]
junc_info = junc_info[junc_info["Cluster"].isin(clusts_keep["Cluster"])]

# break up junction_id column in junc_info into chr, start and end 
junc_info["chr"] = junc_info["junction_id"].str.split("_").str[0]
junc_info["start"] = junc_info["junction_id"].str.split("_").str[1]
junc_info["end"] = junc_info["junction_id"].str.split("_").str[2]
print(len(junc_info["Cluster"].unique()))

In [ ]:
# run function on all clusters to find simple exon skipping events 
clusters_SS = []

for cluster in tqdm(junc_info["Cluster"].unique()):
    clusters_SS.append(check_SS_cluster(cluster))

# keep only entries in clusters_SS that are not None 
clusters_SS = [x for x in clusters_SS if x is not None]
print(len(clusters_SS))

junc_ind_keep = junction_ids_conversion[junction_ids_conversion["Cluster"].isin(clusters_SS)]["junction_id_index"]
final_data = final_data[final_data.junction_id_index.isin(junc_ind_keep)]
final_data.head()

In [ ]:
# update junction_ids_conversion file to only include junctions in clusters_SS 
# update junction_id_index now that we have removed junctions
junction_ids_conversion = junction_ids_conversion[junction_ids_conversion["Cluster"].isin(clusters_SS)]
junction_ids_conversion["new_junction_id_index"] = np.arange(junction_ids_conversion.shape[0])

# print number of unique genes that remain
print(len(junction_ids_conversion.gene_id.unique()))
# print number of unique junctions that remain
print(len(junction_ids_conversion.junction_id.unique()))

In [ ]:
# re-order the remaining junctions and subset the counts matrices
final_data = final_data.merge(junction_ids_conversion, on = "junction_id_index")
final_data.sort_values(by = ["new_junction_id_index"], inplace = True)

to_keep = final_data["junction_id_index"].unique()   # use original junction indices to filter out the count matrices 
junc_counts_sub = junc_counts.tocsr()[:,to_keep].tocoo()
cluster_counts_sub = cluster_counts.tocsr()[:,to_keep].tocoo()

In [ ]:
final_data.iloc[1000]

In [ ]:
# some sanity checks to make sure correct counts are outputted given new indices 
print(junc_counts_sub.toarray()[1577, 1])
print(cluster_counts_sub.toarray()[1577, 1])

### Simulate Data

In [ ]:
simulated_counts, cell_types, cell_type_psi = simulate_junc_counts(cluster_counts_sub, cell_types=cell_ids_conversion.cell_type.astype('category'))

In [ ]:
# turn cell_type_psi into a dataframe wtih columns cell_state1 and cell_state_2 based on index of column 
cell_type_psi_df = pd.DataFrame(cell_type_psi.numpy())
cell_type_psi_df["difference"] = np.diff(cell_type_psi_df)
cell_type_psi_df["new_junction_id_index"] = np.arange(cell_type_psi_df.shape[0])
cell_type_psi_df.head()

In [ ]:
# plot histogram of difference
sns.histplot(cell_type_psi_df["difference"])
# add a title called "Difference in simulated junction p(success) between cell states (K=2)"
plt.title("Difference in simulated junction p(success) between cell states (K=2)")

In [ ]:
sim_juncs_counts = simulated_counts

In [ ]:
# label anything with absolute difference of 0.2 or less as not cell state associated 
cell_type_psi_df["true_label"] = np.where(abs(cell_type_psi_df["difference"]) >= 0.25, "positive", "negative")
cell_type_psi_df.sort_values(by = ["difference"], inplace = True)

In [ ]:
cell_type_psi_df.sort_values(by = ["new_junction_id_index"], inplace = True)
cell_type_psi_df.head()

In [ ]:
junc_ratios = final_data.juncratio
junc_ratios.hist()
# add title that says these are the observed junction usage ratios 
plt.title("Observed junction usage ratios")

In [ ]:
# make dataframe using the following columsn 
sim_junc_counts_flat = pd.DataFrame({"cell_id_index": sim_juncs_counts.row, "new_junction_id_index": sim_juncs_counts.col, "new_junc_count": sim_juncs_counts.data})
sim_junc_counts_flat.head()

# also add new cell type column 
sim_junc_counts_flat["new_cell_type"] = np.array(cell_types[sim_junc_counts_flat["cell_id_index"]])
sim_junc_counts_flat.head()

In [ ]:
# update junction counts in final_data object to be the simulated counts 
final_data = final_data.merge(sim_junc_counts_flat, on = ["cell_id_index", "new_junction_id_index"])
final_data.head()

### Prep simulated data as input into mixture model

In [ ]:
sim_data = final_data.copy() 
# drop the old junction counts and junction id index
sim_data.drop(columns = ["junc_count", "junction_id_index"], inplace = True)
# rename columns new_junction_id_index and new_junc_count to junction_id_index and junc_count
sim_data.rename(columns = {"new_junction_id_index": "junction_id_index", "new_junc_count": "junc_count"}, inplace = True)
sim_data.head()

In [ ]:
# update juncratio 
sim_data["junc_ratio"] = sim_data["junc_count"] / sim_data["cluster_count"]
sim_data.junc_ratio.hist()
# add title saying these are observed junction usage ratios
plt.title("Simulated junction usage ratios")

In [ ]:
sim_data

In [ ]:
cell_index_tensor, junc_index_tensor, my_data = make_torch_data(sim_data, **float_type)

In [ ]:
len(sim_data.junction_id_index.unique())

### Check how well the data is simulated, is there cell type specific splicing?

In [ ]:
cell_type_psi_df.sample(5)

In [ ]:
simple_data = sim_data[["cell_id_index", "cell_type", "junction_id_index", "junc_ratio"]]

In [ ]:
def quick_plot(junc_index):
    simple_data_junc = simple_data[simple_data["junction_id_index"] == junc_index]
    # get PSI 
    psi = cell_type_psi_df[cell_type_psi_df["new_junction_id_index"] == junc_index]["difference"]
    print(psi)
    # make violin plot with jitter 
    print(simple_data_junc.cell_type.value_counts())
    sns.violinplot(data = simple_data_junc, x = "junc_ratio", y = "cell_type")
    # make xlim -1 to 1.1
    plt.xlim(-0.2, 1.2)
    plt.title("Simulated Junction usage ratios for junction {}".format(junc_index))
    plt.show()

In [ ]:
cell_type_psi_df.true_label.value_counts()

In [ ]:
# subset sim_data to just cluster 10071 and cell_id 5457
sim_data[(sim_data["Cluster"] == 10071) & (sim_data["cell_id_index"] == 5457)]

In [ ]:
# let's visualize junction usage ratios for each cell type 
junc = 2557
quick_plot(junc)

### Run mixture model

In [ ]:
K = 2
print(K)

In [ ]:
# make new cell_ids_conversion dataframe using new cell type column
cell_ids_conversion_new = sim_data[["cell_id_index", "cell_type", "cell_id", "new_cell_type"]].drop_duplicates()
cell_ids_conversion_new.head()

In [ ]:
# set random seed
torch.manual_seed(0)

num_trials = 10 # should also be an argument that gets fed in
num_iters = 100 # should also be an argument that gets fed in

# loop over the number of trials (for now just testing using one trial but in general need to evaluate how performance is affected by number of trials)
#reload(betabinomo_mix_singlecells)

start_time = time.time()

# Running for just one K and assessing similarity across trials 

results = [ calculate_CAVI(K, my_data, float_type, hypers, init_labels = None, num_iterations = num_iters) 
           for t in range(num_trials) ]


# write the above line use fstring
print(f"This took {time.time() - start_time} seconds")

In [ ]:
best = np.argmax([ g[-1][-1] for g in results ]) # final ELBO
print(f"The trial with the highest ELBO was {best}")
ALPHA_f, PI_f, GAMMA_f, PHI_f, elbos_all = results[best]
elbos_all = np.array(elbos_all)
plt.plot(elbos_all[1:]); plt.show()

In [ ]:
juncs_probs = ALPHA_f / (ALPHA_f+PI_f)   
 
plt.hist(juncs_probs.cpu().numpy().flatten(), 20)
plt.title('Histogram of learned junction probabilities') 
plt.xlabel('Probability of junction success')
plt.show()

In [ ]:
PHI_f_plot = pd.DataFrame(PHI_f.cpu().numpy())
PHI_f_plot['cell_id'] = cell_ids_conversion["cell_type"].to_numpy()

In [ ]:
# count number of times each cell type is assigned to each cell state
PHI_f_plot.groupby("cell_id").sum()

In [ ]:
cell_ids_conversion_new.sort_values(by = ["cell_id_index"], inplace = True)
cell_ids_conversion_new.head()

In [ ]:
# Obtain cell type labels for every cell in the matrix also 
unique_cell_types = cell_ids_conversion_new['new_cell_type'].unique()
num_unique_types = len(unique_cell_types)
colors = sns.color_palette('Set1', n_colors=num_unique_types)  # You can use any color palette
cell_type_colors = {cell_type: color for cell_type, color in zip(unique_cell_types, colors)}
cell_types = cell_ids_conversion_new.new_cell_type.values
# Convert cell types to corresponding colors for rows and columns
row_colors = [cell_type_colors[cell_type] for cell_type in cell_types]
col_colors = [cell_type_colors[cell_type] for cell_type in cell_types]

In [ ]:
#all_iters_results = check_cell_pairs(results, row_colors, col_colors, cell_type_colors, num_cells_to_plot=100)

In [ ]:
juncs_probs_df = pd.DataFrame(juncs_probs, columns = range(K))
# add "cell_state" to each column name 
juncs_probs_df.columns = ["cell_state_" + str(col) for col in juncs_probs_df.columns]
juncs_probs_df["new_junction_id_index"] = junction_ids_conversion.new_junction_id_index.values
# convert to juncs_probs to pandas dataframe and calculate mean and std across cell states/topics
juncs_probs_df["junction_id"] = junction_ids_conversion.junction_id.values

In [ ]:
juncs_probs_df["learned_diff"] = juncs_probs_df["cell_state_1"] - juncs_probs_df["cell_state_0"]
juncs_probs_df["sim_diff"] = cell_type_psi_df["difference"]
juncs_probs_df.head()

In [ ]:
juncs_probs_df.sort_values(by = ["learned_diff"], inplace = True)
juncs_probs_df.head()

In [ ]:
quick_plot(5442)

In [ ]:
# get spearman correlation between learned and simulated differences
from scipy.stats import spearmanr
spearmanr(juncs_probs_df["learned_diff"], juncs_probs_df["sim_diff"])

In [ ]:
# plot correlation between learned and simulated difference
sns.scatterplot(data = juncs_probs_df, x = "learned_diff", y = "sim_diff")
plt.title("Learned difference vs simulated difference")
plt.show()
# show density where most points lie
sns.kdeplot(data = juncs_probs_df, x = "learned_diff", y = "sim_diff")

In [ ]:
def plot_juncObsUsage(junc_index):

    # print junction ID using junction_ids_conversion
    junc_id = junction_ids_conversion[junction_ids_conversion["new_junction_id_index"] == junc_index].junction_id.values[0]
    print(junc_id)

    # get data for just junc_index 
    junc_dat = sim_data[sim_data.junction_id == junc_id]
    print(junc_dat.new_cell_type.value_counts())
    junc_dat["juncratio"] = junc_dat.junc_count / junc_dat.cluster_count
    print(junc_dat["juncratio"])
    # make violin plot for junc_dat junction usage ratio coloured by cell_type and rotate plot 90 degrees
    plot = ggplot(junc_dat, aes(x='new_cell_type', y='juncratio', fill="new_cell_type")) + geom_violin() + geom_point() + plotnine.labels.ggtitle(junc_id) + plotnine.coords.coord_flip() 

    # add number of cells in each cell_type to plot 
    print(plot)

def plot_juncProbs(junc_index):
    
    # print junction ID using junction_ids_conversion
    print(junction_ids_conversion[junction_ids_conversion["new_junction_id_index"] == junc_index])
    junc_id = junction_ids_conversion[junction_ids_conversion["new_junction_id_index"] == junc_index].junction_id.values[0]
    
    # get data for just junc_index 
    junc_dat = juncs_probs_df[juncs_probs_df.new_junction_id_index == junc_index]
    junc_dat = junc_dat.melt().iloc[0:K]
    junc_dat.value = junc_dat.value.astype(float)
    # make violin plot for junc_dat junction usage ratio coloured by cell_type
    # don't print x-axis tick labels 
    plot = ggplot(junc_dat, aes(x='variable', y='value')) + geom_point() + theme(axis_text_x=element_blank())
    print(plot)

In [ ]:
scores_all_juncs = []
for junc_index in range(juncs_probs.shape[0]):
    a = ALPHA_f[junc_index, ]
    b = PI_f[junc_index, ]
    scores_all_juncs.append(score(a, b).item())

# turn scores_all_juncs into dataframe and add junction_id_index as a column
scores_all_juncs_df = pd.DataFrame(scores_all_juncs, columns = ["score"])
scores_all_juncs_df["new_junction_id_index"] = junction_ids_conversion.new_junction_id_index.values
scores_all_juncs_df.sort_values(by="score", ascending=False).head(10)
juncs_test = scores_all_juncs_df.sort_values(by="score", ascending=False).head(2).new_junction_id_index.values

In [ ]:
scores_all_juncs_df["sim_diff"] = cell_type_psi_df["difference"].values
scores_all_juncs_df.sort_values(by="score", ascending=False).head(10)

In [ ]:
quick_plot(0)

In [ ]:
sim_data.head()

In [ ]:
# for each junction get total sum of junc_count and also report gene_id 
sum_junc_counts = sim_data.groupby(["junction_id_index", "gene_id"]).agg({"junc_count": "sum"}).reset_index()
sum_junc_counts.sort_values(by = ["junction_id_index"], ascending = True, inplace = True)

In [ ]:
sum_junc_counts["difference"] = cell_type_psi_df.difference.values

In [ ]:
sum_junc_counts["score"] = scores_all_juncs_df["score"]
sum_junc_counts.head()

In [ ]:
sum_junc_counts[sum_junc_counts["gene_id"] == "Bpnt1"]

In [ ]:
# get sum of junc counts per gene_id 
sum_junc_counts_gene = sum_junc_counts.groupby(["gene_id"]).agg({"junc_count": "sum", "score": "mean", "difference": "mean"}).reset_index()
# log transform the score and add a small number to avoid taking log of 0
sum_junc_counts_gene["log_score"] = np.log(sum_junc_counts_gene["score"] + 1e-10)

In [ ]:
# make spearman correlation plot between junc_count and score 
sns.scatterplot(data = sum_junc_counts_gene, x = "junc_count", y = "log_score")
plt.title("Spearman correlation between cluster counts and score")
plt.show()
# print the spearmans correlation coefficient
print("Spearman correlation coefficient between junc_count and score: ", sum_junc_counts["junc_count"].corr(sum_junc_counts["score"], method = "spearman"))

In [ ]:
sim_data["new_cell_type"] = sim_data["new_cell_type"].astype(str)

In [ ]:
# for each junction in top10juncs_state1, run plot_juncObsUsage and plot_juncProbs
for junc in juncs_test:
    plot_juncObsUsage(junc)
    plot_juncProbs(junc)

In [ ]:
# convert PHI_f to a dataframe and add a column with cell ID and cell type 
PHI_f = pd.DataFrame(PHI_f)
# Add "CellState" to each column 
PHI_f.columns = ["CellState_" + str(i) for i in range(PHI_f.shape[1])]
PHI_f['cell_id'] = cell_ids_conversion_new.cell_id.values
PHI_f['cell_type'] = cell_ids_conversion_new.new_cell_type.values
PHI_f.groupby('cell_type').sum()

In [ ]:
# group by cell_type and sum across each cellstate 
PHI_f.groupby('cell_type').sum()
sum_prop=PHI_f.groupby('cell_type').sum()/PHI_f.groupby('cell_type').count()
# remove cell_id column 
sum_prop=sum_prop.drop(columns=['cell_id'])
#masked_data = np.ma.masked_equal(sum_prop, 0)
sns.set(font_scale=0.8)  # Adjust font size for labels
# make figure bigger 
plt.figure(figsize=(10, 8))
# make font size of xtickts and yticks bigger
plt.yticks(fontsize=14)
plt.xticks(fontsize=14)
sns.heatmap(sum_prop, annot=True, fmt=".2f", cmap='viridis')

In [ ]:
# evaluate how well junctions are assigned to positive labels (associated with cell state) or negative (no association with cell state)
scores_all_juncs_df.sort_values(by="new_junction_id_index", inplace = True)
scores_all_juncs_df.head()

In [ ]:
# add true_label column to scores_all_juncs_df
scores_all_juncs_df["true_label"] = cell_type_psi_df["true_label"].values
scores_all_juncs_df.head()

In [ ]:
scores_all_juncs_df.shape

In [ ]:
from scipy.special import logit, expit
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import auc

In [ ]:
# make new column where label positive is 1 and negative is 0 in scores_all_juncs_df
scores_all_juncs_df["true_label_num"] = np.where(scores_all_juncs_df["true_label"] == "positive", 1, 0)
scores_all_juncs_df.head()

In [ ]:
scores_all_juncs_df["score"].describe()

In [ ]:
cell_type_psi_df

In [ ]:
# get spearman correlation between score and simulated PSI difference
from scipy.stats import spearmanr
scores_all_juncs_df["difference"] =  cell_type_psi_df["difference"]
# get absolute value of difference
scores_all_juncs_df["difference"] = abs(scores_all_juncs_df["difference"])
# log transform score and add a small value to it 
scores_all_juncs_df["log_score"] = np.log(scores_all_juncs_df["score"] + 1e-10)
print(spearmanr(scores_all_juncs_df["score"], scores_all_juncs_df["difference"]))
# plot scatterplot of score vs difference
sns.scatterplot(data = scores_all_juncs_df, x = "score", y = "difference")

In [ ]:
sim_data.sort_values(by = ["cluster_count"], inplace = True, ascending = False)

In [ ]:
scores_all_juncs_df

In [ ]:
scores_all_juncs_df[scores_all_juncs_df["new_junction_id_index"] == 748]

In [ ]:
scores_all_juncs_df.sort_values(by = ["score"], ascending = False).head(10)

In [ ]:
sim_data.cluster_count.hist()

In [ ]:
# make boxplot of score for each true_label
sns.boxplot(data = scores_all_juncs_df, x = "true_label", y = "score")
plt.title("Boxplot of score for each true_label")
plt.show()

In [ ]:
pre, rec, thres = precision_recall_curve(scores_all_juncs_df["true_label_num"], scores_all_juncs_df["score"])

# Calculate AUC-ROC
auc_roc = roc_auc_score(scores_all_juncs_df["true_label_num"], scores_all_juncs_df["score"])
print("The AUC ROC is: " + str(auc_roc))

auc_pr = auc(rec, pre)
print("the AUC_PR is: " + str(auc_pr))

In [ ]:
# plot precision recall curve using values calculated above
plt.figure(figsize=(10, 5))
plt.plot(rec, pre, color='black')
plt.xlabel('Recall')
plt.ylabel('Precision')